# Hyperparameter Tunnig

This script is using the data pipeline to clean the data.
It will use Hyperopt for hyperparameter tuning and safe the best model via mlflow.

The models will be tested against:
- 1 day
- 1 week
- 2 weeks
- 4 weeks
- 1 quarter
- 2 quarters
- 3 quarters
- 4 quarters

As well as based on data need, this will be evaluated based on CV.

The models to be tuned are:
- SARIMAX
- Tripple Exponential Smoothing
- Prophet
- XG Boost
- Linear Regression
- Random Forest
- LSTM
- Temporal Fusion Transformer (TFT)
- Deep Autoregression Models

# Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import sys
import os
from darts import TimeSeries
from darts.models import Prophet, ARIMA, ExponentialSmoothing
from darts.utils.utils import ModelMode, SeasonalityMode
from darts.metrics import mae, mape, rmse
from hyperopt import hp
import mlflow

# Add the project root to the python path
sys.path.append(os.path.abspath(".."))
from src.processing import DateFeatureTransformer, TimeSeriesWrangler, LagFeatureTransformer, WindowFeatureTransformer
from src.evaluation import DartsObjective, TimeSeriesOptimizer, MLRecursiveObjective, MLOptimizer


/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
mlflow.set_tracking_uri("file:../mlruns")
#mlflow.set_tracking_uri("sqlite:../ipynb/mlflow.db")

# Loading Data

In [3]:
# define path
path = "../data/raw/"

In [4]:
# oil data
oil_df = pd.read_csv(path + "oil.csv")

# Initialize the wrangler
wrangler = TimeSeriesWrangler(
    date_col='date', 
    fill_col='dcoilwtico', 
    freq='D', 
    fill_method='ffill'
)

# Run the cleaning logic
oil = wrangler.clean(oil_df)

In [5]:
# timeseries data
timeseries_df = pd.read_csv(path + "timeseries.csv")

# Initialize the wrangler
wrangler = TimeSeriesWrangler(
    date_col='date', 
    fill_col='unit_sales', 
    freq='D', 
    fill_method='zeros'
)

# Run the cleaning logic
timeseries = wrangler.clean(timeseries_df)

# Variables

In [ ]:
# Defining constants
random_seed = 42
# Change these if you df has different column names
target_col = 'unit_sales'
time_col = 'date'
forecast_horizon = 30
experiment = str(forecast_horizon) + "_day_forecast"
evalsuations = 50
metric_darts = mae
metric_ml = mean_absolute_error
lags_var=[1,2,3,4,5,6,7]
windows_var=[7,14,21]


# SARIMAX

In [7]:
# Now, when you scale, the index stays untouched
from darts.dataprocessing.transformers import Scaler
from sklearn.preprocessing import StandardScaler

# 1. Initialize the scikit-learn scaler you want
base_scaler = StandardScaler()

# 2. Wrap it in the Darts Scaler
transformer = Scaler(scaler=base_scaler)

In [8]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

# Suppress the warning so it doesn't flood your console
warnings.simplefilter('ignore', ConvergenceWarning)

In [9]:
# Defining pipelines for feature engineering
date_pipeline = Pipeline([
    ('date_features', DateFeatureTransformer(column_name=time_col,features=['is_weekend', 'is_holiday', 'is_payday'], payday_val=15, country='EC', drop_date_col=False))
])

timeseries_features = date_pipeline.fit_transform(timeseries)

# Join in the oil data as an exogenous variable
timeseries_oil = timeseries_features.merge(oil, on='date', how='left')

# Set the date as the index first
df = timeseries_oil.set_index(time_col)

# after errors raised fixing oil data via forward fill
if 'dcoilwtico' in timeseries_oil.columns:
    timeseries_oil['dcoilwtico'] = timeseries_oil['dcoilwtico'].ffill().fillna(0)

# Identify where the input breaks (NaNs) and report them in a user-friendly way
nan_report = timeseries_oil.isna().sum()
problematic_cols = nan_report[nan_report > 0]

if not problematic_cols.empty:
    # Build a detailed error message
    error_msg = "\n" + "-"*30 + "\nDATA INTEGRITY BREAKPOINT\n" + "-"*30
    for col, count in problematic_cols.items():
        error_msg += f"\n❌ Column '{col}': {count} missing values ({100*count/len(timeseries_oil):.2f}%)"
    
    # Logic for your oil data: Oil usually lacks weekend data.
    if 'dcoilwtico' in problematic_cols:
        error_msg += "\n\n💡 Pro-tip: Oil prices are often NaN on weekends. Consider forwardfilling."
    
    # Hard stop (The "Breakpoint")
    raise ValueError(error_msg)

# Define the target and exogenous variables
all_exog_features = timeseries_oil.columns.difference([time_col, target_col]).tolist()
series = TimeSeries.from_dataframe(timeseries_oil, time_col=time_col, value_cols=target_col, freq='D')
exog = TimeSeries.from_dataframe(timeseries_oil, time_col=time_col, value_cols=all_exog_features, freq='D')
exog = transformer.fit_transform(exog)

print(timeseries_oil.corr())

# Define your models and search spaces
registry = {
    'SARIMAX': {
        'class': ARIMA,
        'space': {
            # 1. Feature Selection: Toggles each feature on or off
            'selected_features': [hp.choice(f'feat_{f}', [None, f]) for f in all_exog_features],
            
            # 2. Standard SARIMAX Params
            'p': hp.quniform('p', 1, 6, 1),
            'd': 1, #hp.choice('d', [0, 1]),
            'q': hp.quniform('q', 1, 5, 1),
            
            # 3. Seasonal Params (Weekly Seasonality for Ecuador Sales)
            'seasonal_order': (
            hp.quniform('P', 1, 4, 1),
            hp.choice('D', [0, 1]),
            hp.quniform('Q', 1, 4, 1),
            7)#,
            #'trend': hp.choice('trend', ['n', 't'])
        }
    }
}

# Initialize the orchestrator
optimizer = TimeSeriesOptimizer(experiment_name=experiment)

# Run sequentially (Safe for batching)
for name, config in registry.items():
    optimizer.optimize_and_log(
        model_name=name,
        model_class=config['class'],
        space=config['space'],
        series=series,                  # Your Darts TimeSeries
        horizon=forecast_horizon,       # 7-day forecast
        metric=metric_darts,                     # Optimization metric
        exog=exog,                      # Your Darts exogenous TimeSeries
        max_evals=evalsuations                    # Run 50 trials per model
    )



/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/02 07:45:11 INFO mlflow.tracking.fluent: Experiment with name '7_day_forecast' does not exist. Creating a new experiment.


                     date  unit_sales  date_is_weekend  date_is_holiday  \
date             1.000000   -0.010188         0.004833         0.016577   
unit_sales      -0.010188    1.000000         0.685608         0.008411   
date_is_weekend  0.004833    0.685608         1.000000        -0.021108   
date_is_holiday  0.016577    0.008411        -0.021108         1.000000   
date_is_payday  -0.002440   -0.013588         0.013869        -0.044849   
dcoilwtico       0.340182    0.003497         0.006537         0.008804   

                 date_is_payday  dcoilwtico  
date                  -0.002440    0.340182  
unit_sales            -0.013588    0.003497  
date_is_weekend        0.013869    0.006537  
date_is_holiday       -0.044849    0.008804  
date_is_payday         1.000000   -0.008234  
dcoilwtico            -0.008234    1.000000  
Running Hyperopt for SARIMAX up to 30 evals...
  0%|          | 0/30 [00:00<?, ?trial/s, best loss=?]

/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'



Trial failed for ARIMA: LU decomposition error.       
  7%|▋         | 2/30 [11:23<3:06:46, 400.23s/trial, best loss: 186810.99902782627]

/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'



Trial failed for ARIMA: LU decomposition error.                                    
Trial failed for ARIMA: LU decomposition error.                                    
Trial failed for ARIMA: LU decomposition error.                                    
Trial failed for ARIMA: LU decomposition error.                                    
Trial failed for ARIMA: LU decomposition error.                                   
Trial failed for ARIMA: LU decomposition error.                                   
Trial failed for ARIMA: LU decomposition error.                                    
Trial failed for ARIMA: LU decomposition error.                                    
Trial failed for ARIMA: LU decomposition error.                                  
Trial failed for ARIMA: LU decomposition error.                                  
Trial failed for ARIMA: LU decomposition error.                                  
Trial failed for ARIMA: LU decomposition error.                                  
Tr

# Tripple Exponential Smooting

In [10]:
series = TimeSeries.from_dataframe(timeseries, time_col=time_col, value_cols=target_col, freq='D')

# Define your models and search spaces
registry = {
    'Triple Exponential Smoothing': {
        'class': ExponentialSmoothing,
        'space': {
            'trend': hp.choice('trend', [ModelMode.ADDITIVE, ModelMode.MULTIPLICATIVE]),
            'seasonal': hp.choice('seasonal', [SeasonalityMode.ADDITIVE, SeasonalityMode.MULTIPLICATIVE]),
            'damped': hp.choice('damped', [True, False]),
            'seasonal_periods': 7
        }
    }
}

# Initialize the orchestrator
optimizer = TimeSeriesOptimizer(experiment_name=experiment)

# Run sequentially (Safe for batching)
for name, config in registry.items():
    optimizer.optimize_and_log(
        model_name=name,
        model_class=config['class'],
        space=config['space'],
        series=series,                  # Your Darts TimeSeries
        horizon=forecast_horizon,       # 7-day forecast
        metric=metric_darts,                     # Optimization metric
        exog=None,                      # No exogenous variables for ETS
        max_evals=evalsuations                    # Run 50 trials per model
    )



Running Hyperopt for Triple Exponential Smoothing up to 30 evals...
Trial failed for ExponentialSmoothing: endog must be strictly positive when usingmultiplicative trend or seasonal components.
Trial failed for ExponentialSmoothing: endog must be strictly positive when usingmultiplicative trend or seasonal components.
Trial failed for ExponentialSmoothing: endog must be strictly positive when usingmultiplicative trend or seasonal components.
Trial failed for ExponentialSmoothing: endog must be strictly positive when usingmultiplicative trend or seasonal components.
Trial failed for ExponentialSmoothing: endog must be strictly positive when usingmultiplicative trend or seasonal components.
Trial failed for ExponentialSmoothing: endog must be strictly positive when usingmultiplicative trend or seasonal components.
Trial failed for ExponentialSmoothing: endog must be strictly positive when usingmultiplicative trend or seasonal components.
Trial failed for ExponentialSmoothing: endog must 

# Prophet

In [11]:
# Now, when you scale, the index stays untouched
from darts.dataprocessing.transformers import Scaler
from sklearn.preprocessing import StandardScaler

# 1. Initialize the scikit-learn scaler you want
base_scaler = StandardScaler()

# 2. Wrap it in the Darts Scaler
transformer = Scaler(scaler=base_scaler)

In [12]:
# Defining pipelines for feature engineering
date_pipeline = Pipeline([
    ('date_features', DateFeatureTransformer(column_name=time_col,features=['is_weekend', 'is_payday'], payday_val=15, country='EC', drop_date_col=False))
])

timeseries_features = date_pipeline.fit_transform(timeseries)

# Join in the oil data as an exogenous variable
timeseries_oil = timeseries_features.merge(oil, on='date', how='left')

# Set the date as the index first
df = timeseries_oil.set_index(time_col)

# after errors raised fixing oil data via forward fill
if 'dcoilwtico' in timeseries_oil.columns:
    timeseries_oil['dcoilwtico'] = timeseries_oil['dcoilwtico'].ffill().fillna(0)

# Identify where the input breaks (NaNs) and report them in a user-friendly way
nan_report = timeseries_oil.isna().sum()
problematic_cols = nan_report[nan_report > 0]

if not problematic_cols.empty:
    # Build a detailed error message
    error_msg = "\n" + "-"*30 + "\nDATA INTEGRITY BREAKPOINT\n" + "-"*30
    for col, count in problematic_cols.items():
        error_msg += f"\n❌ Column '{col}': {count} missing values ({100*count/len(timeseries_oil):.2f}%)"
    
    # Logic for your oil data: Oil usually lacks weekend data.
    if 'dcoilwtico' in problematic_cols:
        error_msg += "\n\n💡 Pro-tip: Oil prices are often NaN on weekends. Consider forwardfilling."
    
    # Hard stop (The "Breakpoint")
    raise ValueError(error_msg)

# Define the target and exogenous variables
all_exog_features = timeseries_oil.columns.difference([time_col, target_col]).tolist()
series = TimeSeries.from_dataframe(timeseries_oil, time_col=time_col, value_cols=target_col, freq='D')
exog = TimeSeries.from_dataframe(timeseries_oil, time_col=time_col, value_cols=all_exog_features, freq='D')
exog = transformer.fit_transform(exog)

print(timeseries_oil.corr())

# Define your models and search spaces
registry = {
    'Prophet': {
        'class': Prophet,
        'space': {
            # 1. Feature Selection: Toggles each feature on or off
            'selected_features': [hp.choice(f'feat_{f}', [None, f]) for f in all_exog_features],
            
            # 2. Standard Prophet Params
            'changepoint_prior_scale': hp.loguniform('changepoint_prior_scale', np.log(0.001), np.log(0.5)),
            'seasonality_prior_scale': hp.loguniform('seasonality_prior_scale', np.log(0.01), np.log(10.0)),
            'holidays_prior_scale': hp.loguniform('holidays_prior_scale', np.log(0.01), np.log(10.0)),
            'seasonality_mode': hp.choice('seasonality_mode', ['additive', 'multiplicative']),
            'changepoint_range': hp.uniform('changepoint_range', 0.8, 0.95)
        }
    }
}

# Initialize the orchestrator
optimizer = TimeSeriesOptimizer(experiment_name=experiment)

# Run sequentially (Safe for batching)
for name, config in registry.items():
    optimizer.optimize_and_log(
        model_name=name,
        model_class=config['class'],
        space=config['space'],
        series=series,                  # Your Darts TimeSeries
        horizon=forecast_horizon,       # 7-day forecast
        metric=metric_darts,                     # Optimization metric
        exog=exog,                      # Your Darts exogenous TimeSeries
        max_evals=evalsuations                    # Run 50 trials per model
    )



                     date  unit_sales  date_is_weekend  date_is_payday  \
date             1.000000   -0.010188         0.004833       -0.002440   
unit_sales      -0.010188    1.000000         0.685608       -0.013588   
date_is_weekend  0.004833    0.685608         1.000000        0.013869   
date_is_payday  -0.002440   -0.013588         0.013869        1.000000   
dcoilwtico       0.340182    0.003497         0.006537       -0.008234   

                 dcoilwtico  
date               0.340182  
unit_sales         0.003497  
date_is_weekend    0.006537  
date_is_payday    -0.008234  
dcoilwtico         1.000000  
Running Hyperopt for Prophet up to 30 evals...
  0%|          | 0/30 [00:00<?, ?trial/s, best loss=?]

21:02:19 - cmdstanpy - INFO - Chain [1] start processing

21:02:19 - cmdstanpy - INFO - Chain [1] done processing

21:02:19 - cmdstanpy - INFO - Chain [1] start processing

21:02:19 - cmdstanpy - INFO - Chain [1] done processing

21:02:19 - cmdstanpy - INFO - Chain [1] start processing

21:02:20 - cmdstanpy - INFO - Chain [1] done processing

21:02:20 - cmdstanpy - INFO - Chain [1] start processing

21:02:20 - cmdstanpy - INFO - Chain [1] done processing

21:02:20 - cmdstanpy - INFO - Chain [1] start processing

21:02:20 - cmdstanpy - INFO - Chain [1] done processing

21:02:20 - cmdstanpy - INFO - Chain [1] start processing

21:02:20 - cmdstanpy - INFO - Chain [1] done processing

21:02:20 - cmdstanpy - INFO - Chain [1] start processing

21:02:20 - cmdstanpy - INFO - Chain [1] done processing

21:02:20 - cmdstanpy - INFO - Chain [1] start processing

21:02:20 - cmdstanpy - INFO - Chain [1] done processing

21:02:20 - cmdstanpy - INFO - Chain [1] start processing

21:02:20 - cmdstanpy -

  3%|▎         | 1/30 [00:05<02:53,  5.98s/trial, best loss: 97.5157176125276]

21:02:25 - cmdstanpy - INFO - Chain [1] start processing

21:02:25 - cmdstanpy - INFO - Chain [1] done processing

21:02:25 - cmdstanpy - INFO - Chain [1] start processing

21:02:25 - cmdstanpy - INFO - Chain [1] done processing

21:02:25 - cmdstanpy - INFO - Chain [1] start processing

21:02:25 - cmdstanpy - INFO - Chain [1] done processing

21:02:25 - cmdstanpy - INFO - Chain [1] start processing

21:02:25 - cmdstanpy - INFO - Chain [1] done processing

21:02:25 - cmdstanpy - INFO - Chain [1] start processing

21:02:25 - cmdstanpy - INFO - Chain [1] done processing

21:02:26 - cmdstanpy - INFO - Chain [1] start processing

21:02:26 - cmdstanpy - INFO - Chain [1] done processing

21:02:26 - cmdstanpy - INFO - Chain [1] start processing

21:02:26 - cmdstanpy - INFO - Chain [1] done processing

21:02:26 - cmdstanpy - INFO - Chain [1] start processing

21:02:26 - cmdstanpy - INFO - Chain [1] done processing

21:02:26 - cmdstanpy - INFO - Chain [1] start processing

21:02:26 - cmdstanpy -

  7%|▋         | 2/30 [00:12<02:50,  6.08s/trial, best loss: 96.96987718987666]

...

 97%|█████████▋| 29/30 [03:51<00:06,  6.18s/trial, best loss: 96.42669912143846]

21:06:11 - cmdstanpy - INFO - Chain [1] start processing

21:06:11 - cmdstanpy - INFO - Chain [1] done processing

21:06:11 - cmdstanpy - INFO - Chain [1] start processing

21:06:11 - cmdstanpy - INFO - Chain [1] done processing

21:06:11 - cmdstanpy - INFO - Chain [1] start processing

21:06:11 - cmdstanpy - INFO - Chain [1] done processing

21:06:11 - cmdstanpy - INFO - Chain [1] start processing

21:06:11 - cmdstanpy - INFO - Chain [1] done processing

21:06:11 - cmdstanpy - INFO - Chain [1] start processing

21:06:11 - cmdstanpy - INFO - Chain [1] done processing

21:06:11 - cmdstanpy - INFO - Chain [1] start processing

21:06:11 - cmdstanpy - INFO - Chain [1] done processing

21:06:11 - cmdstanpy - INFO - Chain [1] start processing

21:06:11 - cmdstanpy - INFO - Chain [1] done processing

21:06:11 - cmdstanpy - INFO - Chain [1] start processing

21:06:11 - cmdstanpy - INFO - Chain [1] done processing

21:06:11 - cmdstanpy - INFO - Chain [1] start processing

21:06:11 - cmdstanpy -

100%|██████████| 30/30 [03:57<00:00,  7.91s/trial, best loss: 96.42669912143846]

21:06:17 - cmdstanpy - INFO - Chain [1] start processing

21:06:17 - cmdstanpy - INFO - Chain [1] done processing




Logging Champion Prophet to MLflow...


# XG Boost

In [13]:
# Defining pipelines for feature engineering
date_pipeline = Pipeline([
    ('date_features', DateFeatureTransformer(column_name=time_col,features=['is_weekend', 'is_payday', 'is_holiday', 'month', 'year', 'day_of_week'], payday_val=15, country='EC', drop_date_col=False)),
    ('lag_features', LagFeatureTransformer({target_col: lags_var}, fill_method='bfill')),
    ('window_features', WindowFeatureTransformer({target_col: windows_var}, fill_method='bfill'))
])

timeseries_features = date_pipeline.fit_transform(timeseries)

# Join in the oil data as an exogenous variable
timeseries_oil = timeseries_features.merge(oil, on='date', how='left')

# after errors raised fixing oil data via forward fill
if 'dcoilwtico' in timeseries_oil.columns:
    timeseries_oil['dcoilwtico'] = timeseries_oil['dcoilwtico'].ffill().fillna(0)

# Identify where the input breaks (NaNs) and report them in a user-friendly way
nan_report = timeseries_oil.isna().sum()
problematic_cols = nan_report[nan_report > 0]

if not problematic_cols.empty:
    # Build a detailed error message
    error_msg = "\n" + "-"*30 + "\nDATA INTEGRITY BREAKPOINT\n" + "-"*30
    for col, count in problematic_cols.items():
        error_msg += f"\n❌ Column '{col}': {count} missing values ({100*count/len(timeseries_oil):.2f}%)"
    
    # Logic for your oil data: Oil usually lacks weekend data.
    if 'dcoilwtico' in problematic_cols:
        error_msg += "\n\n💡 Pro-tip: Oil prices are often NaN on weekends. Consider forwardfilling."
    
    # Hard stop (The "Breakpoint")
    raise ValueError(error_msg)

# Define the target and exogenous variables
all_exog_features = timeseries_oil.columns.difference([time_col, target_col]).tolist()

print(timeseries_oil.corr())

# Define your models and search spaces
registry = {
    'XGBoost': {
        'class': xgb.XGBRegressor,
        'space': {
            # 1. Feature Selection (Dynamic Toggling)
            'selected_features': [hp.choice(f'feat_{f}', [None, f]) for f in all_exog_features],
            
            # 2. Structural Hyperparameters
            'n_estimators': hp.quniform('n_estimators', 100, 1000, 10), # Number of trees
            'max_depth': hp.quniform('max_depth', 3, 10, 1),           # Depth of trees
            'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.2)),
            
            # 3. Regularization (Crucial to prevent leakage-driven overfitting)
            'subsample': hp.uniform('subsample', 0.6, 0.9),            # Row sampling
            'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 0.9), # Feature sampling
            'gamma': hp.uniform('gamma', 0, 5),                        # Minimum loss reduction
            'reg_alpha': hp.loguniform('reg_alpha', np.log(1e-8), np.log(1.0)), # L1
            'reg_lambda': hp.loguniform('reg_lambda', np.log(1e-8), np.log(1.0)), # L2
            
            # 4. XGBoost Specifics
            'random_state': random_seed,
            'n_jobs': -1,
            'objective': 'reg:absoluteerror'
        }
    }
}

# Initialize the orchestrator
optimizer = MLOptimizer(experiment_name=experiment)

# Run sequentially (Safe for batching)
for name, config in registry.items():
    optimizer.optimize_ml_model(
            model_name=name,
            model_class=config['class'],
            space=config['space'],
            df=timeseries_oil,         # Your raw pandas DF
            target_col=target_col,
            date_col=time_col,
            lag_list=lags_var,
            rolling_list=windows_var,
            metric=metric_ml,
            start_ratio=0.7, 
            step_size=forecast_horizon,
            max_evals=evalsuations
        )



                                    date  unit_sales  date_is_weekend  \
date                        1.000000e+00   -0.010188         0.004833   
unit_sales                 -1.018818e-02    1.000000         0.685608   
date_is_weekend             4.833203e-03    0.685608         1.000000   
date_is_payday             -2.439893e-03   -0.013588         0.013869   
date_is_holiday             1.657718e-02    0.008411        -0.021108   
date_month                  2.740324e-01   -0.014658         0.003627   
date_year                   6.905224e-01    0.007055         0.002800   
date_day_of_week           -9.968524e-16    0.504302         0.790395   
unit_sales_lag_1           -1.108140e-02    0.229155         0.126814   
unit_sales_lag_2           -1.602371e-02   -0.234608        -0.379759   
unit_sales_lag_3           -2.203985e-02   -0.162257        -0.291924   
unit_sales_lag_4           -2.530515e-02   -0.197525        -0.217520   
unit_sales_lag_5           -2.427723e-02   -0.18775

2026/05/02 22:13:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Successfully logged XGBoost with min loss: 95.2627


# Linear Regression

In [14]:
# Defining pipelines for feature engineering
date_pipeline = Pipeline([
    ('date_features', DateFeatureTransformer(column_name=time_col,features=['is_weekend', 'is_payday', 'is_holiday', 'month', 'year', 'day_of_week'], payday_val=15, country='EC', drop_date_col=False)),
    ('lag_features', LagFeatureTransformer({target_col: lags_var}, fill_method='bfill')),
    ('window_features', WindowFeatureTransformer({target_col: windows_var}, fill_method='bfill'))
])

timeseries_features = date_pipeline.fit_transform(timeseries)

# Join in the oil data as an exogenous variable
timeseries_oil = timeseries_features.merge(oil, on='date', how='left')

# after errors raised fixing oil data via forward fill
if 'dcoilwtico' in timeseries_oil.columns:
    timeseries_oil['dcoilwtico'] = timeseries_oil['dcoilwtico'].ffill().fillna(0)

# Identify where the input breaks (NaNs) and report them in a user-friendly way
nan_report = timeseries_oil.isna().sum()
problematic_cols = nan_report[nan_report > 0]

if not problematic_cols.empty:
    # Build a detailed error message
    error_msg = "\n" + "-"*30 + "\nDATA INTEGRITY BREAKPOINT\n" + "-"*30
    for col, count in problematic_cols.items():
        error_msg += f"\n❌ Column '{col}': {count} missing values ({100*count/len(timeseries_oil):.2f}%)"
    
    # Logic for your oil data: Oil usually lacks weekend data.
    if 'dcoilwtico' in problematic_cols:
        error_msg += "\n\n💡 Pro-tip: Oil prices are often NaN on weekends. Consider forwardfilling."
    
    # Hard stop (The "Breakpoint")
    raise ValueError(error_msg)

# Define the target and exogenous variables
all_exog_features = timeseries_oil.columns.difference([time_col, target_col]).tolist()

print(timeseries_oil.corr())

# Define your models and search spaces
registry = {
    'ElasticNet': {
    'class': ElasticNet,
    'space': {
        # 1. Feature Selection 
        # (Requires custom wrapper to pop this key and filter X before training!)
        'selected_features': [hp.choice(f'lr_feat_{f}', [None, f]) for f in all_exog_features],

        # 2. Regularization Strength
        # Expanded the upper bound. Sometimes collinear data needs heavy shrinkage.
        'alpha': hp.loguniform('alpha', np.log(1e-5), np.log(100.0)),

        # 3. The L1/L2 Mix - FIXED
        # Strictly > 0 to prevent coordinate descent collapse
        'l1_ratio': hp.uniform('l1_ratio', 0.01, 1.0),

        # 4. Structural Parameters
        'fit_intercept': hp.choice('fit_intercept', [True, False]),
        
        # 5. Solver Mechanics
        # Allowing Hyperopt to find the right balance of iteration limits
        'max_iter': hp.quniform('max_iter', 1000, 5000, 500), 
        'tol': hp.loguniform('tol', np.log(1e-5), np.log(1e-3)),
        
        'random_state': random_seed
    }
}
}

# Initialize the orchestrator
optimizer = MLOptimizer(experiment_name=experiment)

# Run sequentially (Safe for batching)
for name, config in registry.items():
    optimizer.optimize_ml_model(
            model_name=name,
            model_class=config['class'],
            space=config['space'],
            df=timeseries_oil,         # Your raw pandas DF
            target_col=target_col,
            date_col=time_col,
            lag_list=lags_var,
            rolling_list=windows_var,
            metric=metric_ml,
            start_ratio=0.7, 
            step_size=forecast_horizon,
            max_evals=evalsuations
        )



                                    date  unit_sales  date_is_weekend  \
date                        1.000000e+00   -0.010188         0.004833   
unit_sales                 -1.018818e-02    1.000000         0.685608   
date_is_weekend             4.833203e-03    0.685608         1.000000   
date_is_payday             -2.439893e-03   -0.013588         0.013869   
date_is_holiday             1.657718e-02    0.008411        -0.021108   
date_month                  2.740324e-01   -0.014658         0.003627   
date_year                   6.905224e-01    0.007055         0.002800   
date_day_of_week           -9.968524e-16    0.504302         0.790395   
unit_sales_lag_1           -1.108140e-02    0.229155         0.126814   
unit_sales_lag_2           -1.602371e-02   -0.234608        -0.379759   
unit_sales_lag_3           -2.203985e-02   -0.162257        -0.291924   
unit_sales_lag_4           -2.530515e-02   -0.197525        -0.217520   
unit_sales_lag_5           -2.427723e-02   -0.18775

/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.026e+06, tolerance: 4.342e+03
  model = cd_fast.enet_coordinate_descent(

/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.073e+06, tolerance: 4.454e+03
  model = cd_fast.enet_coordinate_descent(

/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might wan

 17%|█▋        | 5/30 [00:01<00:06,  3.93trial/s, best loss: 99.7690473951783]

/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.105e+06, tolerance: 5.751e+03
  model = cd_fast.enet_coordinate_descent(

/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.148e+06, tolerance: 5.864e+03
  model = cd_fast.enet_coordinate_descent(

/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might wan

 23%|██▎       | 7/30 [00:01<00:04,  4.82trial/s, best loss: 99.7690473951783]

...

 63%|██████▎   | 19/30 [00:04<00:02,  3.75trial/s, best loss: 98.40457573862304]

/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.058e+06, tolerance: 4.501e+03
  model = cd_fast.enet_coordinate_descent(

/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.077e+06, tolerance: 4.579e+03
  model = cd_fast.enet_coordinate_descent(

/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might wan

100%|██████████| 30/30 [00:06<00:00,  5.00trial/s, best loss: 96.82403433663465]

/opt/homebrew/Caskroom/miniforge/base/envs/ml_timeseries_env/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.302e+06, tolerance: 4.134e+02
  model = cd_fast.enet_coordinate_descent(
2026/05/02 22:13:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 22:13:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Logging Champion ElasticNet to MLflow...
Successfully logged ElasticNet with min loss: 96.8240


# Random Forest

In [15]:
# Defining pipelines for feature engineering
date_pipeline = Pipeline([
    ('date_features', DateFeatureTransformer(column_name=time_col,features=['is_weekend', 'is_payday', 'is_holiday', 'month', 'year', 'day_of_week'], payday_val=15, country='EC', drop_date_col=False)),
    ('lag_features', LagFeatureTransformer({target_col: lags_var}, fill_method='bfill')),
    ('window_features', WindowFeatureTransformer({target_col: windows_var}, fill_method='bfill'))
])

timeseries_features = date_pipeline.fit_transform(timeseries)

# Join in the oil data as an exogenous variable
timeseries_oil = timeseries_features.merge(oil, on='date', how='left')

# after errors raised fixing oil data via forward fill
if 'dcoilwtico' in timeseries_oil.columns:
    timeseries_oil['dcoilwtico'] = timeseries_oil['dcoilwtico'].ffill().fillna(0)

# Identify where the input breaks (NaNs) and report them in a user-friendly way
nan_report = timeseries_oil.isna().sum()
problematic_cols = nan_report[nan_report > 0]

if not problematic_cols.empty:
    # Build a detailed error message
    error_msg = "\n" + "-"*30 + "\nDATA INTEGRITY BREAKPOINT\n" + "-"*30
    for col, count in problematic_cols.items():
        error_msg += f"\n❌ Column '{col}': {count} missing values ({100*count/len(timeseries_oil):.2f}%)"
    
    # Logic for your oil data: Oil usually lacks weekend data.
    if 'dcoilwtico' in problematic_cols:
        error_msg += "\n\n💡 Pro-tip: Oil prices are often NaN on weekends. Consider forwardfilling."
    
    # Hard stop (The "Breakpoint")
    raise ValueError(error_msg)

# Define the target and exogenous variables
all_exog_features = timeseries_oil.columns.difference([time_col, target_col]).tolist()

print(timeseries_oil.corr())

# Define your models and search spaces
registry = {
        'RandomForest': {
        'class': RandomForestRegressor, 
        'space': {
            # 1. Feature Selection 
            # (Requires the same custom wrapper to pop this key and filter X!)
            'selected_features': [hp.choice(f'rf_feat_{f}', [None, f]) for f in all_exog_features],

            # 2. Forest Mechanics
            # 100 to 1000 is generally the sweet spot. Beyond 1000, you hit severe 
            # diminishing returns on performance vs. compute time.
            # NOTE: hp.quniform returns floats. Your wrapper MUST cast this to int!
            'n_estimators': hp.quniform('n_estimators', 100, 1000, 50),

            # 3. Tree Structural Regularization
            # 'None' allows trees to grow fully (often fine for RF if bagging is strong),
            # but bounding it prevents severe overfitting on highly noisy datasets.
            'max_depth': hp.quniform('max_depth', 5, 50, 1),
            
            # Minimum samples to split. Higher values prevent the model from 
            # learning highly specific, noisy relationships.
            'min_samples_split': hp.quniform('min_samples_split', 2, 20, 1),
            
            # Minimum samples per leaf. Bumping this > 1 drastically smooths the 
            # predictions, which is incredibly useful for regression tasks.
            'min_samples_leaf': hp.quniform('min_samples_leaf', 1, 20, 1),

            # 4. Feature Subsampling (The "Random" in Random Forest)
            # 'sqrt' is classic for Classification; None or fractional (0.3 - 0.5) 
            # often works better for Regression. 
            # (Note: 'auto' is deprecated/removed in modern scikit-learn).
            'max_features': hp.choice('max_features', ['sqrt', 'log2', None, 0.3, 0.5, 0.8]),

            # 6. System Parameters
            'n_jobs': -1, # Maximize core usage
            'random_state': random_seed
        }
    }
}


# Initialize the orchestrator
optimizer = MLOptimizer(experiment_name=experiment)

# Run sequentially (Safe for batching)
for name, config in registry.items():
    optimizer.optimize_ml_model(
            model_name=name,
            model_class=config['class'],
            space=config['space'],
            df=timeseries_oil,         # Your raw pandas DF
            target_col=target_col,
            date_col=time_col,
            lag_list=lags_var,
            rolling_list=windows_var,
            metric=metric_ml,
            start_ratio=0.7, 
            step_size=forecast_horizon,
            max_evals=evalsuations
        )



                                    date  unit_sales  date_is_weekend  \
date                        1.000000e+00   -0.010188         0.004833   
unit_sales                 -1.018818e-02    1.000000         0.685608   
date_is_weekend             4.833203e-03    0.685608         1.000000   
date_is_payday             -2.439893e-03   -0.013588         0.013869   
date_is_holiday             1.657718e-02    0.008411        -0.021108   
date_month                  2.740324e-01   -0.014658         0.003627   
date_year                   6.905224e-01    0.007055         0.002800   
date_day_of_week           -9.968524e-16    0.504302         0.790395   
unit_sales_lag_1           -1.108140e-02    0.229155         0.126814   
unit_sales_lag_2           -1.602371e-02   -0.234608        -0.379759   
unit_sales_lag_3           -2.203985e-02   -0.162257        -0.291924   
unit_sales_lag_4           -2.530515e-02   -0.197525        -0.217520   
unit_sales_lag_5           -2.427723e-02   -0.18775

2026/05/02 22:37:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 22:37:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully logged RandomForest with min loss: 92.7003


# LSTM

# TFT

# Deep Autoregression Models